In [2]:
# Experiment 4: Tokenizer lab - BPE vs WordPiece vs SentencePiece on English, Telugu and Hindi
from transformers import AutoTokenizer
from google.colab import userdata
from google import genai
import pandas as pd

texts = {
    "English": "The morning breeze carried the fresh scent of rain across the quiet garden.",
    "Telugu":  "నిశ్శబ్దంగా ఉన్న తోటలో ఉదయపు గాలి తాజా వర్షపు సువాసనను మోసుకొచ్చింది.",
    "Hindi":   "शांत बगीचे में सुबह की हवा बारिश की ताज़ा खुशबू लेकर आई।",
}
tokenizers = {
    "BPE (gpt2)": "gpt2",
    "WordPiece (bert-base-multilingual-cased)": "bert-base-multilingual-cased",
    "SentencePiece (xlm-roberta-base)": "xlm-roberta-base",
}

rows = []
for name, model_id in tokenizers.items():
    tok = AutoTokenizer.from_pretrained(model_id)
    for lang, s in texts.items():
        ids = tok(s, add_special_tokens=False).input_ids
        rows.append({"tokenizer": name, "language": lang, "words": len(s.split()),
                     "tokens": len(ids), "tokens_per_word": round(len(ids) / len(s.split()), 2)})
    print(name, "| vocab size:", tok.vocab_size)
    print("   English pieces:", tok.tokenize(texts["English"])[:10])
    print("   Telugu pieces :", tok.tokenize(texts["Telugu"])[:10])
    print("   Token IDs (Hindi):", tok(texts["Hindi"], add_special_tokens=False).input_ids[:10])

# Hosted model's own tokenizer (Gemini) for comparison
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
for lang, s in texts.items():
    n = client.models.count_tokens(model="gemini-3.1-flash-lite", contents=s).total_tokens
    rows.append({"tokenizer": "Gemini (gemini-3.1-flash-lite)", "language": lang, "words": len(s.split()),
                 "tokens": n, "tokens_per_word": round(n / len(s.split()), 2)})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Token-ratio table (documented result): how many times more tokens than English
ratio = df.pivot(index="tokenizer", columns="language", values="tokens")
ratio["Telugu / English"] = (ratio["Telugu"] / ratio["English"]).round(2)
ratio["Hindi / English"] = (ratio["Hindi"] / ratio["English"]).round(2)
print("\nToken-ratio table:")
print(ratio.to_string())

BPE (gpt2) | vocab size: 50257
   English pieces: ['The', 'Ġmorning', 'Ġbreeze', 'Ġcarried', 'Ġthe', 'Ġfresh', 'Ġscent', 'Ġof', 'Ġrain', 'Ġacross']
   Telugu pieces : ['à', '°', '¨', 'à', '°', '¿', 'à', '°', '¶', 'à']
   Token IDs (Hindi): [11976, 114, 48077, 11976, 224, 11976, 97, 28225, 105, 11976]
WordPiece (bert-base-multilingual-cased) | vocab size: 119547
   English pieces: ['The', 'morning', 'br', '##ee', '##ze', 'carried', 'the', 'fresh', 's', '##cent']
   Telugu pieces : ['న', '##ి', '##శ', '##్', '##శ', '##బ్', '##దం', '##గా', 'ఉన్న', 'తో']
   Token IDs (Hindi): [896, 56563, 887, 34315, 19030, 10532, 898, 14070, 18351, 17110]
SentencePiece (xlm-roberta-base) | vocab size: 250002
   English pieces: ['▁The', '▁morning', '▁bre', 'eze', '▁carried', '▁the', '▁fresh', '▁scen', 't', '▁of']
   Telugu pieces : ['▁ని', 'శ్', 'శబ్ద', 'ంగా', '▁ఉన్న', '▁తో', 'ట', 'లో', '▁ఉదయ', 'పు']
   Token IDs (Hindi): [112078, 127644, 97718, 421, 82158, 471, 49868, 139196, 471, 185048]
                